# فصل ۱۴: دانش خارجی و معماری RAG (External Knowledge & RAG)
## بخش ۱: معماری RAG و پیاده‌سازی با LCEL (RAG Architecture & LCEL)

### 🏛️ تمثیل بنیادین: آزمون حفظی در برابر آزمون کتاب‌باز (Open-Book Exam)
تا این لحظه، ما با مدل‌های زبانی بزرگ (LLM) مانند یک **«دانشمند همه‌چیزدان اما منزوی»** برخورد می‌کردیم:
1. **محدودیت تاریخ انقضا (Knowledge Cutoff Date):** دانش این دانشمند در یک تاریخ مشخص متوقف شده است و از وقایع امروز کاملاً بی‌خبر است.
2. **بی‌خبری از داده‌های خصوصی:** هیچ اطلاعی از اسناد داخلی شرکت، فاکتورها، مکاتبات، کاتالوگ‌های اختصاصی یا فایل‌های شخصی شما ندارد.
3. **توهم خطرناک (Hallucination):** اگر پاسخی را نداند، برخلاف انسان اعتراف به ندانستن نمی‌کند؛ بلکه با کمال اعتمادبه‌نفس داستانی دروغین سر هم می‌کند!

**معماری RAG یعنی تبدیل آزمون از «حفظی» به «کتاب‌باز»!**
به جای اینکه مدل مجبور باشد همه چیز را حفظ کند، در لحظه آزمون کتابخانه اختصاصی ما (Vector Database) را باز می‌کند، صفحه مرتبط را بیرون می‌کشد و پاسخ مستند را تولید می‌کند.

### 🧩 کالبدشکافی ارکان سه‌گانه RAG:

نام **RAG** از ۳ کلمه بنیادین سرچشمه می‌گیرد:
- **۱. بازیابی (Retrieval):** پیدا کردن تکه‌های مرتبط اسناد از پایگاه داده برداری بر اساس سوال کاربر.
- **۲. افزودن / غنی‌سازی (Augmentation):** تزریق اسناد بازیابی‌شده به پرامپت به عنوان زمینه و کانتکست (`Context`).
- **۳. تولید (Generation):** ارسال پرامپت تقویت‌شده به مدل زبانی جهت نگارش پاسخ دقیق، مستند و بدون توهم.

### 🔄 دیاگرام گردش کار (End-to-End Workflow):

```mermaid
flowchart LR
    A["👤 سوال کاربر (Query)"] --> B["🔢 تبدیل به بردار (Embed Query)"]
    B --> C["🗄️ جستجوی برداری (Vector DB)"]
    C --> D["📄 بازیابی k چانک برتر"]
    D --> E["📝 ساخت پرامپت (Prompt Construction)"]
    A --> E
    E --> F["🤖 استنتاج مدل (LLM Generation)"]
    F --> G["🎯 پاسخ نهایی مستند"]
```

### ۱. آماده‌سازی محیط و بارگذاری داده‌های پایه در دیتابیس برداری

ابتدا دیتابیس برداری (Chroma) را راه‌اندازی می‌کنیم و اسناد پایه‌ای فصل قبل (دانش فضایی و سیارات) به همراه داده‌های گزارش سازمانی را در آن درج می‌کنیم:

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

# فعال‌سازی حالت آفلاین برای مدل امبدینگ لوکال
os.environ["HF_HUB_OFFLINE"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# ۱. مدل تعبیه برداری چندزبانه
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# ۲. اسناد دانش اختصاصی برای پایگاه برداری
knowledge_docs = [
    Document(
        page_content="مریخ چهارمین سیاره از خورشید است و به دلیل وجود اکسید آهن در سطح آن، به سیاره سرخ معروف است.",
        metadata={"topic": "astronomy", "target": "mars"}
    ),
    Document(
        page_content="مشتری بزرگترین سیاره منظومه شمسی است و یک غول گازی با طوفان‌های عظیم مانند لکه سرخ بزرگ می‌باشد.",
        metadata={"topic": "astronomy", "target": "jupiter"}
    ),
    Document(
        page_content="زهره گرم‌ترین سیاره منظومه شمسی است و اتمسفر بسیار غلیظی از کربن دی‌اکسید دارد.",
        metadata={"topic": "astronomy", "target": "venus"}
    ),
    Document(
        page_content="گزارش مالی سال ۱۴۰۲ نشان‌دهنده سود خالص ۳۵ میلیارد تومانی شرکت با رشد ۴۰ درصدی نسبت به سال قبل است.",
        metadata={"topic": "finance", "year": 1402}
    )
]

# ۳. ایجاد پایگاه برداری Chroma
vector_db = Chroma.from_documents(
    documents=knowledge_docs,
    embedding=embedding_model,
    collection_name="my_course_collection"
)

print(f"تعداد {len(knowledge_docs)} سند با موفقیت در پایگاه برداری ذخیره شدند.")

Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1518.22it/s]


تعداد 4 سند با موفقیت در پایگاه برداری ذخیره شدند.


### ۲. قدم اول: تبدیل دیتابیس به بازیاب (`Retriever`)

یک پایگاه داده به خودی خود فقط یک مخزن ذخیره‌سازی است. برای اینکه بتواند در خط لوله مدرن LangChain شرکت کند، باید با متد `.as_retriever()` به یک **Runnable** بازیاب تبدیل شود.
آرگومان `search_kwargs={"k": 3}` تضمین می‌کند که همیشه ۳ سند با بیشترین شباهت کسینوسی برگردانده شوند:

In [2]:
# تبدیل به موتور جستجوی اسناد (Retriever)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# تست اولیه بازیاب با یک سوال نمونه
test_retrieved = retriever.invoke("مریخ")
print(f"تعداد اسناد بازیابی‌شده برای 'مریخ': {len(test_retrieved)}")
print(f"متن سند برتر: {test_retrieved[0].page_content}")

تعداد اسناد بازیابی‌شده برای 'مریخ': 3
متن سند برتر: مریخ چهارمین سیاره از خورشید است و به دلیل وجود اکسید آهن در سطح آن، به سیاره سرخ معروف است.


### ۳. قدم دوم: طراحی پرامپت استاندارد RAG (Prompt Engineering)

طراحی پرامپت حساس‌ترین بخش است! ما باید به طور اکید به مدل فرمان دهیم که:
1. **تنها و تنها بر اساس متن کانتکست (`{context}`)** پاسخ دهد.
2. اگر پاسخ در کانتکست وجود نداشت، **صادقانه بگوید "I don't know" (نمی‌دانم)** و از خود جوابی اختراع نکند.

In [3]:
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context:
{context}

Question: {question}
If you don't know the answer based on the context, simply say "I don't know". Do not make up an answer.
"""

prompt = ChatPromptTemplate.from_template(template)
print("الگوی پرامپت RAG با موفقیت ساخته شد.")

الگوی پرامپت RAG با موفقیت ساخته شد.


### ۴. قدم سوم: راه‌اندازی مدل زبانی جمنای (Google Gemini Setup)

مدل زبانی جمنای را با مدل فعال و معتبر **`gemini-flash-lite-latest`** با `temperature=0` مقداردهی می‌کنیم:

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

# راه‌اندازی مدل جمنای با دمای صفر برای دقت استنتاج حداکثری
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0
)

print("مدل جمنای با موفقیت آماده به کار شد:", llm.model)

مدل جمنای با موفقیت آماده به کار شد: gemini-flash-lite-latest


### ۵. قدم چهارم: ساخت زنجیره RAG با زبان بیانی لانگ‌چین (LCEL)

در LangChain مدرن، کل خط لوله RAG با استفاده از **عملگر پایپ (`|`)** در LCEL ساخته می‌شود:

```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
```

**اتفاقاتی که در خط لوله رخ می‌دهد:**
1. متغیر `question` با `RunnablePassthrough()` مستقیماً از ورودی کاربر دریافت می‌شود.
2. ورودی کاربر همزمان به `retriever` ارسال شده و اسناد بازیابی‌شده توسط تابع `format_docs` به یک متن یکپارچه تبدیل می‌شوند.
3. دیکشنری `{'context': ..., 'question': ...}` وارد `prompt` شده و پرامپت نهایی ساخته می‌شود.
4. پرامپت به `llm` داده شده و با `StrOutputParser` خروجی به صورت رشته متنی خالص تحویل داده می‌شود.

In [5]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    """اسناد بازیابی شده را به یک رشته متنی طولانی تبدیل می‌کند."""
    return "\n\n".join([d.page_content for d in docs])

# تعریف زنجیره RAG با مدل جمنای
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("زنجیره کامل LCEL RAG با موفقیت کامپایل شد.")

زنجیره کامل LCEL RAG با موفقیت کامپایل شد.


### ۶. قدم پنجم: اجرای سیستم و راستی‌آزمایی عدم توهم با مدل جمنای

بیایید دو حالت را آزمایش کنیم:
1. **سوال موجود در دیتابیس:** *«مریخ چه رنگی است؟»*
2. **سوال ناموجود (تست عدم توهم):** *«قیمت دلار امروز چقدر است؟»*

In [6]:
# ۱. تست سوالی که پاسخ آن در دیتابیس است
q1 = "مریخ چه رنگی است؟"
resp1 = rag_chain.invoke(q1)
print(f"❓ سوال ۱: {q1}")
print(f"💡 پاسخ جمنای: {resp1}\n")

# ۲. تست سوال خارج از دانش دیتابیس (عدم توهم)
q2 = "قیمت دلار امروز چقدر است؟"
resp2 = rag_chain.invoke(q2)
print(f"❓ سوال ۲: {q2}")
print(f"🛡️ پاسخ جمنای (عدم توهم): {resp2}")

❓ سوال ۱: مریخ چه رنگی است؟
💡 پاسخ جمنای: بر اساس متن، مریخ به دلیل وجود اکسید آهن در سطح آن، **سرخ** (سیاره سرخ) است.

❓ سوال ۲: قیمت دلار امروز چقدر است؟
🛡️ پاسخ جمنای (عدم توهم): I don't know


### ⚠️ چرا معماری Naive RAG در مقیاس صنعتی کافی نیست؟

سیستمی که تا اینجا ساختیم به **Naive RAG (سیستم RAG ساده‌لوح)** معروف است. در پروژه‌های واقعی و دیتابیس‌های چند هزار صفحه‌ای، این سیستم ساده با ۳ آسیب‌پذیری حیاتی مواجه می‌شود:

#### آسیب‌پذیری اول: خطای دقت و انحراف معنایی (Semantic Drift & Ambiguity)
اگر کاربر کلمه «محدودیت‌های سیستم» را در یک پایگاه داده فنی جستجو کند، تشابه برداری ممکن است به جای محدودیت‌های فنی سرور، چانکی از «محدودیت‌های حساب بانکی» را بازیابی کند صرفاً به دلیل اشتراک کلمه «محدودیت»!

برای رفع این چالش، سیستم‌های پیشرفته از **فیلتر متادیتا (Metadata Filtering)** استفاده می‌کنند:

In [7]:
# شبیه‌سازی اسناد چند حوزه‌ای با اشتراک کلمات کلیدی
multi_domain_docs = [
    Document(
        page_content="محدودیت‌های سیستم مالی: سقف برداشت روزانه حساب‌ها ۵۰ میلیون تومان است.",
        metadata={"category": "finance"}
    ),
    Document(
        page_content="محدودیت‌های سیستم زیرساخت: حداکثر بار پردازش سرور ۱۰۰۰ درخواست در ثانیه است.",
        metadata={"category": "technical"}
    )
]

domain_db = Chroma.from_documents(multi_domain_docs, embedding=embedding_model)

# جستجوی ساده‌لوحانه (ممکن است سند مالی اول شود)
naive_res = domain_db.similarity_search("محدودیت‌های سیستم", k=1)
print("بازیابی ساده‌لوحانه (Naive):\n", naive_res[0].page_content)

# بازیابی دقیق با فیلتر متادیتا (فقط اسناد فنی)
filtered_res = domain_db.similarity_search("محدودیت‌های سیستم", k=1, filter={"category": "technical"})
print("\nبازیابی هوشمند و هدایت‌شده با متادیتا:\n", filtered_res[0].page_content)

بازیابی ساده‌لوحانه (Naive):
 محدودیت‌های سیستم زیرساخت: حداکثر بار پردازش سرور ۱۰۰۰ درخواست در ثانیه است.

بازیابی هوشمند و هدایت‌شده با متادیتا:
 محدودیت‌های سیستم زیرساخت: حداکثر بار پردازش سرور ۱۰۰۰ درخواست در ثانیه است.


#### آسیب‌پذیری دوم: گم شدن در میانه (Lost in the Middle)
پژوهش‌های هوش مصنوعی (مانند مقاله معروف استنفورد) نشان داده‌اند که وقتی تعداد اسناد بازیابی‌شده زیاد است (مثلاً ۱۰ چانک)، مدل‌های زبانی عمدتاً به **ابتدای پرامپت** و **انتهای پرامپت** توجه می‌کنند.
اگر پاسخ اصلی سوال در سند شماره ۵ (وسط کانتکست) باشد، مدل آن را نادیده گرفته و به خطا می‌افتد!

In [8]:
# شبیه‌سازی پدیده Lost in the Middle
chunks = [f"سند {i}: این یک متن عمومی درباره استانداردهای اداری است." for i in range(1, 11)]
# قرار دادن پاسخ کلیدی دقیقاً در چانک ۵ (وسط)
chunks[4] = "سند ۵ (نکته حیاتی): رمز ورود اضطراری به اتاق سرور 9942 می باشد."

middle_context = "\n\n".join(chunks)
print("توزیع اطلاعات در پرامپت:")
print(f"- ابتدا: {chunks[0]}")
print(f"- میانه (محل آسیب‌پذیری): {chunks[4]}")
print(f"- انتها: {chunks[-1]}")

توزیع اطلاعات در پرامپت:
- ابتدا: سند 1: این یک متن عمومی درباره استانداردهای اداری است.
- میانه (محل آسیب‌پذیری): سند ۵ (نکته حیاتی): رمز ورود اضطراری به اتاق سرور 9942 می باشد.
- انتها: سند 10: این یک متن عمومی درباره استانداردهای اداری است.


#### آسیب‌پذیری سوم: افزونگی و کپی‌های تکراری (Redundancy)
اگر دیتابیس ۳ سند بازیابی کند و هر ۳ سند جملات مشابهی را تکرار کنند (مثلاً ۳ نسخه از اعلامیه شرکت)، پنجره زمینه پر می‌شود بدون اینکه هیچ اطلاعات متنوع یا تکمیلی به مدل داده شود.

In [9]:
redundant_chunks = [
    "ساعت کاری شرکت در ماه رمضان از ۹ صبح تا ۱۴ است.",
    "اعلامیه منابع انسانی: ساعات کاری در ماه مبارک رمضان ۹ الی ۱۴ می‌باشد.",
    "بخشنامه داخلی: پرسنل گرامی ساعت کاری رمضان ۹ تا ۲ بعدازظهر است."
]

print("اسناد افزونه (Redundant) که کانتکست را بیهوده اشغال می‌کنند:")
for i, c in enumerate(redundant_chunks, 1):
    print(f"{i}. {c}")

اسناد افزونه (Redundant) که کانتکست را بیهوده اشغال می‌کنند:
1. ساعت کاری شرکت در ماه رمضان از ۹ صبح تا ۱۴ است.
2. اعلامیه منابع انسانی: ساعات کاری در ماه مبارک رمضان ۹ الی ۱۴ می‌باشد.
3. بخشنامه داخلی: پرسنل گرامی ساعت کاری رمضان ۹ تا ۲ بعدازظهر است.


### 🎯 جمع‌بندی فصل ۱۴، درسنامه اول:

1. **RAG:** تلفیق هوشمند بازیابی اطلاعات، غنی‌سازی زمینه و استنتاج بدون توهم.
2. **LCEL:** ساخت زنجیره با پایپ و ترکیب ماژولار `retriever | format_docs` با `prompt` و `llm`.
3. **تضمین دقت:** کنترل توهم با پرامپت مقیدساز (`Answer only based on context...`).
4. **محدودیت‌های Naive RAG:** شناخت پدیده‌های Semantic Drift، Lost in the Middle و Redundancy که در درسنامه‌های بعدی این فصل با تکنیک‌های پیشرفته (Reranking, MMR, Contextual Compression, Hybrid Search) درمان خواهند شد!